### Notebook 09 : End-to-End Multi-Agent Orchestration

##### 1. Notebook Purpose

This notebook connects all previously developed agents into one complete multi-agent workflow.

The workflow:

- Receives the user request.
- Creates the initial shared state.
- Sends the request to the Coordinator Agent.
- Reads the execution plan created by the Coordinator.
- Identifies which tasks are ready.
- Executes the appropriate agents.
- Stores agent results in shared state.
- Continues until all planned tasks are complete.
- Returns the final grounded response.
- 
The orchestrator does not perform SQL analysis, prediction, vector search, retention logic, or response generation itself.

It only coordinates the agents.

##### 2. Technologies Used

- Python
- Databricks
- Pydantic
- TypedDict
- Callable type hints
- Dependency-based task orchestration
- Shared multi-agent state
- Agent runner registry


##### 3. Input

The notebook accepts a natural-language user request.

Example:

``` text

user_request = (
    "For customer 1001, predict churn risk, review similar customer notes, recommend a retention action, and provide a final response."
)
```

##### 4. Output

The workflow returns the completed MultiAgentState.

The final state contains information such as:

``` text

{
    "user_request": ...,
    "execution_plan": ...,
    "agent_results": ...,
    "execution_history": ...,
    "errors": ...,
    "final_response": ...,
}

```

##### 5. Architecture

``` text

User Request
      │
      ▼
create_initial_state()
      │
      ▼
Coordinator Agent
      │
      ▼
Execution Plan
      │
      ▼
Orchestrator
      │
      ├── SQL Agent
      ├── Prediction Agent
      ├── Vector Search Agent
      ├── Retention Agent
      └── Final Response Agent
      │
      ▼
Completed Shared State
      │
      ▼
Final Grounded Response

```

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

##### 6. Load Shared Components

In [0]:
%run ./01_shared_models

In [0]:
%run ./02_shared_state_and_helpers

##### 7. Load Agent Notebooks

In [0]:
%run ./03_coordinator_agent

In [0]:
print(
    "After 03:",
    "run_coordinator_agent"
    if "run_coordinator_agent" in globals()
    else "run_coordinator_agent NOT FOUND"
)

In [0]:
%run ./04_sql_agent

In [0]:
print(
    "After 04 - coordinator:",
    "FOUND"
    if "run_coordinator_agent" in globals()
    else "NOT FOUND"
)

print(
    "After 04 - sql agent:",
    "FOUND"
    if "run_sql_agent" in globals()
    else "NOT FOUND"
)

print(
    "After 04 - sql tool:",
    "FOUND"
    if "sql_analytics_tool" in globals()
    else "NOT FOUND"
)

In [0]:
%run ./05_prediction_agent

In [0]:
print(
    "After 05 - coordinator:",
    "FOUND"
    if "run_coordinator_agent" in globals()
    else "NOT FOUND"
)

print(
    "After 05 - sql agent:",
    "FOUND"
    if "run_sql_agent" in globals()
    else "NOT FOUND"
)

print(
    "After 05 - prediction agent:",
    "FOUND"
    if "run_prediction_agent" in globals()
    else "NOT FOUND"
)

print(
    "After 05 - prediction tool:",
    "FOUND"
    if "prediction_tool" in globals()
    else "NOT FOUND"
)

In [0]:
%run ./06_vector_search_agent

In [0]:
for name in [
    "run_coordinator_agent",
    "run_sql_agent",
    "sql_analytics_tool",
    "run_prediction_agent",
    "prediction_tool",
    "run_vector_search_agent",
    "vector_search_tool",
]:
    print(
        name,
        "->",
        "FOUND" if name in globals()
        else "NOT FOUND",
    )

In [0]:
%run ./07_retention_agent

In [0]:
%run ./08_final_response_agent

In [0]:
required_names = [
    "run_coordinator_agent",
    "run_sql_agent",
    "run_prediction_agent",
    "run_vector_search_agent",
    "run_retention_agent",
    "run_final_response_agent",
    "sql_analytics_tool",
    "prediction_tool",
    "vector_search_tool",
    "retention_tool",
    "invoke_llm",
]

for name in required_names:
    print(
        name,
        "->",
        "FOUND" if name in globals()
        else "NOT FOUND",
    )

In [0]:
required_names = [
    "run_coordinator_agent",
    "run_sql_agent",
    "run_prediction_agent",
    "run_vector_search_agent",
    "run_retention_agent",
    "run_final_response_agent",
    "sql_analytics_tool",
    "prediction_tool",
    "vector_search_tool",
    "retention_tool",
    "invoke_llm",
]

for name in required_names:
    print(
        name,
        "->",
        "FOUND" if name in globals()
        else "NOT FOUND",
    )

##### 8. Imports

In [0]:
from typing import Any, Callable, Dict, List, Tuple
import inspect

##### 9. Agent Function Types

In [0]:
AgentFunction = Callable[
    ...,
    MultiAgentState,
]

In [0]:
AgentRunnerConfig = Tuple[
    AgentFunction,
    Dict[str, Any],
]

##### 10. Agent Runner Registry

In [0]:
def create_agent_runners(
    sql_analytics_tool: SQLAnalyticsFunction,
    prediction_tool: PredictionToolFunction,
    vector_search_tool: VectorSearchToolFunction,
    retention_tool: RetentionToolFunction,
    llm_invoke: LLMInvokeFunction,
) -> Dict[str, AgentRunnerConfig]:
    """
    Create the agent-runner registry using
    injected tools and dependencies.
    """

    return {
        SQL_AGENT_NAME: (
            run_sql_agent,
            {
                "sql_analytics_tool": (
                    sql_analytics_tool
                ),
            },
        ),

        PREDICTION_AGENT_NAME: (
            run_prediction_agent,
            {
                "prediction_tool": (
                    prediction_tool
                ),
            },
        ),

        VECTOR_SEARCH_AGENT_NAME: (
            run_vector_search_agent,
            {
                "vector_search_tool": (
                    vector_search_tool
                ),
                "num_results": (
                    DEFAULT_NUM_RESULTS
                ),
            },
        ),

        RETENTION_AGENT_NAME: (
            run_retention_agent,
            {
                "retention_tool": (
                    retention_tool
                ),
            },
        ),

        FINAL_RESPONSE_AGENT_NAME: (
            run_final_response_agent,
            {
                "llm_invoke": (
                    llm_invoke
                ),
            },
        ),
    }

##### 11. Check Whether a Task Is Completed

In [0]:
def is_task_completed(
    state: MultiAgentState,
    task: AgentTask,
) -> bool:
    """
    Check whether an agent task has completed.

    A task is considered completed when its agent
    has stored a successful result in Shared State.
    """

    agent_result = state[
        "agent_results"
    ].get(task.agent_name)

    if agent_result is None:
        return False

    return (
        agent_result.status == "success"
    )

##### 12. Check Whether a Dependency Succeeded

In [0]:
def did_dependency_succeed(
    state: MultiAgentState,
    dependency_name: str,
) -> bool:
    """
    Check whether a dependency produced
    a successful agent result.
    """

    dependency_result = state[
        "agent_results"
    ].get(dependency_name)

    if dependency_result is None:
        return False

    return (
        dependency_result.status == "success"
    )

##### 13. Check Whether a Task Is Ready

In [0]:
def is_task_ready(
    state: MultiAgentState,
    task: AgentTask,
) -> bool:
    """
    Check whether all dependencies required by
    a task have completed successfully.
    """

    for dependency_name in task.depends_on:

        if not did_dependency_succeed(
            state=state,
            dependency_name=dependency_name,
        ):
            return False

    return True

##### 14. Get All Ready Tasks

In [0]:
def get_ready_tasks(
    state: MultiAgentState,
) -> List[AgentTask]:
    """
    Return all execution-plan tasks that:

    1. Have not already completed.
    2. Have all required dependencies satisfied.
    """

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        return []

    execution_plan = (
        coordinator_result.execution_plan
    )

    ready_tasks: List[AgentTask] = []

    for task in execution_plan:

        if is_task_completed(
            state=state,
            task=task,
        ):
            continue

        if is_task_ready(
            state=state,
            task=task,
        ):
            ready_tasks.append(task)

    return ready_tasks

##### 16. Run One Agent

In [0]:
def run_agent(
    state: MultiAgentState,
    task: AgentTask,
    agent_runners: Dict[
        str,
        AgentRunnerConfig,
    ],
) -> MultiAgentState:
    """
    Execute one agent task using the function and
    keyword arguments stored in the agent registry.
    """

    agent_name = task.agent_name

    if agent_name not in agent_runners:
        error_message = (
            "No agent runner is registered for "
            f"agent '{agent_name}'."
        )

        record_agent_error(
            state=state,
            agent_name=ORCHESTRATOR_NAME,
            error_code=(
                "AGENT_REGISTRATION_ERROR"
            ),
            error_message=error_message,
        )

        raise ValueError(
            error_message
        )

    agent_function, agent_kwargs = (
        agent_runners[agent_name]
    )

    return agent_function(
        state=state,
        **agent_kwargs,
    )

##### 17. Check Whether the Workflow Is Complete

In [0]:
def is_workflow_complete(
    state: MultiAgentState,
) -> bool:
    """
    Check whether every task in the Coordinator's
    execution plan has completed successfully.
    """

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        return False

    execution_plan = (
        coordinator_result.execution_plan
    )

    if not execution_plan:
        return False

    return all(
        is_task_completed(
            state=state,
            task=task,
        )
        for task in execution_plan
    )

##### 18. Get Incomplete Task Names

In [0]:
def get_incomplete_agent_names(
    state: MultiAgentState,
) -> List[str]:
    """
    Return agent names for tasks that have not
    completed successfully.
    """

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        return []

    execution_plan = (
        coordinator_result.execution_plan
    )

    return [
        task.agent_name
        for task in execution_plan
        if not is_task_completed(
            state=state,
            task=task,
        )
    ]

##### 19. Main Multi-Agent Workflow

In [0]:
def run_multi_agent_workflow(
    user_request: str,
    coordinator_llm_invoke: LLMInvokeFunction,
    final_response_llm_invoke: LLMInvokeFunction,
    sql_analytics_tool: SQLAnalyticsFunction,
    prediction_tool: PredictionToolFunction,
    vector_search_tool: VectorSearchToolFunction,
    retention_tool: RetentionToolFunction,
    max_iterations: int = 20,
    debug: bool = False,
) -> MultiAgentState:
    """
    Run the complete dependency-based multi-agent workflow.

    Workflow:
    1. Create the initial shared state.
    2. Run the Coordinator Agent.
    3. Read the Coordinator execution plan.
    4. Create the agent-runner registry.
    5. Find ready tasks.
    6. Execute ready tasks.
    7. Stop immediately if an executed agent fails.
    8. Repeat until all tasks complete successfully.

    Parameters
    ----------
    user_request:
        Original natural-language user request.

    llm_invoke:
        Injected LLM function used by the Coordinator
        and Final Response Agent.

    sql_analytics_tool:
        Real SQL Analytics Tool.

    prediction_tool:
        Real Prediction Tool.

    vector_search_tool:
        Real Vector Search Tool.

    retention_tool:
        Real Retention Tool.

    max_iterations:
        Safety limit that prevents infinite loops.

    debug:
        When True, print orchestration details
        for troubleshooting.
    """

    # =========================================================
    # Validate inputs
    # =========================================================

    if not user_request or not user_request.strip():
        raise ValueError(
            "The user request cannot be empty."
        )

    if max_iterations <= 0:
        raise ValueError(
            "max_iterations must be greater than zero."
        )

    # =========================================================
    # Create initial shared state
    # =========================================================

    state = create_initial_state(
        user_request=user_request
    )

    # =========================================================
    # Run Coordinator
    # =========================================================

    state = run_coordinator_agent(
        state=state,
        llm_invoke=coordinator_llm_invoke,
    )

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        error_message = (
            "The Coordinator Agent did not create "
            "a coordinator result."
        )

        record_agent_error(
            state=state,
            agent_name=ORCHESTRATOR_NAME,
            error_code="MISSING_COORDINATOR_RESULT",
            error_message=error_message,
        )

        raise ValueError(
            error_message
        )

    if coordinator_result.status != "success":
        error_message = (
            "The Coordinator Agent did not complete "
            "successfully."
        )

        record_agent_error(
            state=state,
            agent_name=ORCHESTRATOR_NAME,
            error_code="COORDINATOR_FAILED",
            error_message=error_message,
        )

        raise RuntimeError(
            error_message
        )

    execution_plan = (
        coordinator_result.execution_plan
    )

    if not execution_plan:
        error_message = (
            "The Coordinator Agent did not create "
            "an execution plan."
        )

        record_agent_error(
            state=state,
            agent_name=ORCHESTRATOR_NAME,
            error_code="MISSING_EXECUTION_PLAN",
            error_message=error_message,
        )

        raise ValueError(
            error_message
        )

    # =========================================================
    # Create agent-runner registry
    # =========================================================

    agent_runners = create_agent_runners(
        sql_analytics_tool=sql_analytics_tool,
        prediction_tool=prediction_tool,
        vector_search_tool=vector_search_tool,
        retention_tool=retention_tool,
        llm_invoke=final_response_llm_invoke,
    )

    # =========================================================
    # Main orchestration loop
    # =========================================================

    iteration = 0

    while not is_workflow_complete(
        state=state
    ):

        iteration += 1

        if debug:
            print()
            print("=" * 80)
            print(
                f"ORCHESTRATION ITERATION {iteration}"
            )
            print("=" * 80)

            print(
                "Current successful agent results:",
                list(
                    state["agent_results"].keys()
                ),
            )

            print(
                "Current errors:",
                state["errors"],
            )

        # =====================================================
        # Maximum-iteration protection
        # =====================================================

        if iteration > max_iterations:
            error_message = (
                "The workflow exceeded the maximum "
                "number of iterations: "
                f"{max_iterations}."
            )

            record_agent_error(
                state=state,
                agent_name=ORCHESTRATOR_NAME,
                error_code=(
                    "MAXIMUM_ITERATIONS_EXCEEDED"
                ),
                error_message=error_message,
            )

            raise RuntimeError(
                error_message
            )

        # =====================================================
        # Find currently ready tasks
        # =====================================================

        ready_tasks = get_ready_tasks(
            state=state
        )

        if debug:
            print(
                "Ready tasks:",
                [
                    task.agent_name
                    for task in ready_tasks
                ],
            )

        # =====================================================
        # Detect blocked workflow
        # =====================================================

        if not ready_tasks:

            incomplete_agents = (
                get_incomplete_agent_names(
                    state=state
                )
            )

            error_message = (
                "The workflow cannot continue because "
                "no incomplete task is currently ready. "
                "This may indicate a missing dependency, "
                "a failed dependency, or a circular "
                "dependency. "
                f"Incomplete agents: "
                f"{incomplete_agents}"
            )

            record_agent_error(
                state=state,
                agent_name=ORCHESTRATOR_NAME,
                error_code="BLOCKED_WORKFLOW",
                error_message=error_message,
            )

            raise RuntimeError(
                error_message
            )

        # =====================================================
        # Execute ready tasks
        # =====================================================

        for task in ready_tasks:

            if debug:
                print()
                print(
                    f"Running agent: "
                    f"{task.agent_name}"
                )

            errors_before = len(
                state["errors"]
            )

            state = run_agent(
                state=state,
                task=task,
                agent_runners=agent_runners,
            )

            errors_after = len(
                state["errors"]
            )

            if debug:
                print(
                    "Agent results after execution:",
                    list(
                        state[
                            "agent_results"
                        ].keys()
                    ),
                )

                print(
                    "Errors after execution:",
                    state["errors"],
                )

            # =================================================
            # Stop immediately if this execution added an error
            # =================================================

            if errors_after > errors_before:

                latest_error = (
                    state["errors"][-1]
                )

                error_message = (
                    f"Agent '{task.agent_name}' "
                    "failed during orchestration: "
                    f"{latest_error.error_message}"
                )

                record_agent_error(
                    state=state,
                    agent_name=ORCHESTRATOR_NAME,
                    error_code=(
                        "AGENT_EXECUTION_FAILED"
                    ),
                    error_message=error_message,
                )

                raise RuntimeError(
                    error_message
                )

            # =================================================
            # Verify the agent actually completed
            # =================================================

            agent_result = state[
                "agent_results"
            ].get(task.agent_name)

            if agent_result is None:

                error_message = (
                    f"Agent '{task.agent_name}' "
                    "finished without storing a "
                    "successful result."
                )

                record_agent_error(
                    state=state,
                    agent_name=ORCHESTRATOR_NAME,
                    error_code=(
                        "AGENT_RESULT_MISSING"
                    ),
                    error_message=error_message,
                )

                raise RuntimeError(
                    error_message
                )

            if agent_result.status != "success":

                error_message = (
                    f"Agent '{task.agent_name}' "
                    "did not complete successfully."
                )

                record_agent_error(
                    state=state,
                    agent_name=ORCHESTRATOR_NAME,
                    error_code=(
                        "AGENT_RESULT_NOT_SUCCESSFUL"
                    ),
                    error_message=error_message,
                )

                raise RuntimeError(
                    error_message
                )

    # =========================================================
    # Workflow completed
    # =========================================================

    if debug:
        print()
        print("=" * 80)
        print("WORKFLOW COMPLETED SUCCESSFULLY")
        print("=" * 80)

        print(
            "Completed agents:",
            list(
                state["agent_results"].keys()
            ),
        )

    return state

##### 20. How the Main Loop Works

###### Initial state:

state["agent_results"] = {}

The Coordinator creates a plan such as:

``` text

Prediction Agent
    depends_on = []

Vector Search Agent
    depends_on = []

Retention Agent
    depends_on =
        Prediction Agent
        Vector Search Agent

Final Response Agent
    depends_on =
        Retention Agent

```

###### Iteration 1

``` text

ready_tasks = [
    prediction_task,
    vector_search_task,
]

```

The loop executes them sequentially:

``` text

for task in ready_tasks:
    state = run_agent(
        state=state,
        task=task,
    )

```

After execution:

``` text

state["agent_results"] = {
    PREDICTION_AGENT_NAME: prediction_result,
    VECTOR_SEARCH_AGENT_NAME: vector_search_result,
}

```

###### Iteration 2


Retention dependencies are now satisfied:

``` text

ready_tasks = [
    retention_task,
]

```

After Retention executes:

``` text

state["agent_results"] = {
    PREDICTION_AGENT_NAME: prediction_result,
    VECTOR_SEARCH_AGENT_NAME: vector_search_result,
    RETENTION_AGENT_NAME: retention_result,
}
```

###### Iteration 3

``` text

The Final Response task becomes ready:

ready_tasks = [
    final_response_task,
]
```
After it executes, every task is complete and the loop ends.

##### 21. End-to-End Test Helper

In [0]:
def display_workflow_results(
    final_state: MultiAgentState,
) -> None:
    """
    Display the main outputs from a completed workflow.
    """

    print("=" * 100)
    print("USER REQUEST")
    print("=" * 100)
    print(
        final_state.get(
            "user_request",
            "Not available",
        )
    )

    print("\n" + "=" * 100)
    print("EXECUTION PLAN")
    print("=" * 100)

    for task in final_state.get(
        "execution_plan",
        [],
    ):
        print(
            f"Agent: {task.agent_name}"
        )

        print(
            f"Dependencies: {task.depends_on}"
        )

        print("-" * 100)

    print("\n" + "=" * 100)
    print("EXECUTION HISTORY")
    print("=" * 100)

    execution_history = final_state.get(
        "execution_history",
        [],
    )

    if execution_history:
        for record in execution_history:
            print(record)
    else:
        print(
            "No execution-history records found."
        )

    print("\n" + "=" * 100)
    print("AGENT RESULTS")
    print("=" * 100)

    agent_results = final_state.get(
        "agent_results",
        {},
    )

    if agent_results:
        for agent_name, result in agent_results.items():
            print(
                f"\nAgent: {agent_name}"
            )

            print(
                f"Result: {result}"
            )
    else:
        print(
            "No agent results found."
        )

    print("\n" + "=" * 100)
    print("ERRORS")
    print("=" * 100)

    errors = final_state.get(
        "errors",
        [],
    )

    if errors:
        for error in errors:
            print(error)
    else:
        print(
            "No workflow errors."
        )

    print("\n" + "=" * 100)
    print("FINAL RESPONSE")
    print("=" * 100)

    final_response_result = final_state[
    "agent_results"
    ].get(FINAL_RESPONSE_AGENT_NAME)

    if (
        isinstance(
            final_response_result,
            FinalResponseAgentResult,
        )
        and final_response_result.status == "success"
        and final_response_result.final_response
    ):
        print(
            final_response_result.final_response
        )
    else:
        print(
            "No final response was generated."
        )

##### 22. Unit test

In [0]:
def test_multi_agent_orchestrator_unit_test() -> None:
    """
    Unit test the dependency-based multi-agent
    orchestration logic without calling real tools,
    model endpoints, Vector Search, or LLMs.

    Tests:
    1. Combined workflow dependency ordering.
    2. Simple SQL workflow.
    3. Blocked workflow.
    4. Maximum-iteration protection.
    """

    # =========================================================
    # Shared test helper
    # =========================================================

    def create_coordinator_result(
        request_type: RequestType,
        execution_plan: List[AgentTask],
    ) -> CoordinatorResult:
        """
        Create a successful mock Coordinator result.
        """

        return CoordinatorResult(
            agent_name=COORDINATOR_AGENT_NAME,
            status="success",
            message=(
                "Execution plan created successfully."
            ),
            task_description=(
                "Plan the multi-agent workflow."
            ),
            error=None,
            request_type=request_type,
            reasoning=(
                "Mock reasoning for orchestrator "
                "unit testing."
            ),
            execution_plan=execution_plan,
        )

    # =========================================================
    # Mock result helpers
    # =========================================================

    def create_prediction_result(
    ) -> PredictionAgentResult:

        return PredictionAgentResult(
            agent_name=PREDICTION_AGENT_NAME,
            status="success",
            message=(
                "Prediction completed successfully."
            ),
            task_description=(
                "Predict customer churn."
            ),
            error=None,
            customer_id="7590-VHVEG",
            predicted_category="Churn",
            confidence=None,
            model_name="mock_model",
            raw_prediction=True,
        )

    def create_vector_search_result(
    ) -> VectorSearchAgentResult:

        return VectorSearchAgentResult(
            agent_name=VECTOR_SEARCH_AGENT_NAME,
            status="success",
            message=(
                "Vector Search completed successfully."
            ),
            task_description=(
                "Find similar customer notes."
            ),
            error=None,
            task_id="task_2",
            query=(
                "Find customer service concerns."
            ),
            results=[
                VectorSearchItem(
                    customer_id="8388-DMKAE",
                    note=(
                        "Customer reported repeated "
                        "connectivity problems."
                    ),
                    similarity_score=0.90,
                )
            ],
        )

    def create_retention_result(
    ) -> RetentionAgentResult:

        return RetentionAgentResult(
            agent_name=RETENTION_AGENT_NAME,
            status="success",
            message=(
                "Retention recommendation completed."
            ),
            task_description=(
                "Recommend a retention action."
            ),
            error=None,
            task_id="task_3",
            customer_id="7590-VHVEG",
            recommended_action=(
                "service_quality_review"
            ),
            action_reason=(
                "The customer is predicted to churn "
                "and supporting notes indicate "
                "service-quality concerns."
            ),
            prediction_label="Churn",
            prediction_confidence=None,
            supporting_notes=[
                (
                    "Customer reported repeated "
                    "connectivity problems."
                )
            ],
        )

    def create_final_response_result(
    ) -> FinalResponseAgentResult:

        return FinalResponseAgentResult(
            agent_name=FINAL_RESPONSE_AGENT_NAME,
            status="success",
            message=(
                "Final Response Agent completed "
                "successfully."
            ),
            task_description=(
                "Generate the final grounded response."
            ),
            error=None,
            final_response=(
                "A service-quality review is "
                "recommended for the customer."
            ),
        )

    # =========================================================
    # TEST 1
    # Combined workflow dependency ordering
    # =========================================================

    print("=" * 80)
    print(
        "TEST 1: Combined workflow dependency ordering"
    )
    print("=" * 80)

    state = create_initial_state(
        "For customer 7590-VHVEG, predict churn, "
        "review customer notes, recommend a retention "
        "action, and provide a final response."
    )

    state["coordinator_result"] = (
        create_coordinator_result(
            request_type="combined",
            execution_plan=[
                AgentTask(
                    task_id="task_1",
                    agent_name=(
                        PREDICTION_AGENT_NAME
                    ),
                    task_description=(
                        "Predict churn for customer "
                        "7590-VHVEG."
                    ),
                    depends_on=[],
                ),
                AgentTask(
                    task_id="task_2",
                    agent_name=(
                        VECTOR_SEARCH_AGENT_NAME
                    ),
                    task_description=(
                        "Find relevant customer notes."
                    ),
                    depends_on=[],
                ),
                AgentTask(
                    task_id="task_3",
                    agent_name=(
                        RETENTION_AGENT_NAME
                    ),
                    task_description=(
                        "Recommend a retention action."
                    ),
                    depends_on=[
                        PREDICTION_AGENT_NAME,
                        VECTOR_SEARCH_AGENT_NAME,
                    ],
                ),
                AgentTask(
                    task_id="task_4",
                    agent_name=(
                        FINAL_RESPONSE_AGENT_NAME
                    ),
                    task_description=(
                        "Generate the final grounded "
                        "response."
                    ),
                    depends_on=[
                        RETENTION_AGENT_NAME,
                    ],
                ),
            ],
        )
    )

    # ---------------------------------------------------------
    # Initially Prediction + Vector Search should be ready
    # ---------------------------------------------------------

    ready_tasks = get_ready_tasks(
        state=state
    )

    ready_agent_names = {
        task.agent_name
        for task in ready_tasks
    }

    assert ready_agent_names == {
        PREDICTION_AGENT_NAME,
        VECTOR_SEARCH_AGENT_NAME,
    }

    # ---------------------------------------------------------
    # Complete Prediction only
    # Retention should still NOT be ready
    # ---------------------------------------------------------

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_prediction_result()

    ready_tasks = get_ready_tasks(
        state=state
    )

    ready_agent_names = {
        task.agent_name
        for task in ready_tasks
    }

    assert ready_agent_names == {
        VECTOR_SEARCH_AGENT_NAME
    }

    # ---------------------------------------------------------
    # Complete Vector Search
    # Retention should now become ready
    # ---------------------------------------------------------

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_vector_search_result()

    ready_tasks = get_ready_tasks(
        state=state
    )

    ready_agent_names = {
        task.agent_name
        for task in ready_tasks
    }

    assert ready_agent_names == {
        RETENTION_AGENT_NAME
    }

    # ---------------------------------------------------------
    # Complete Retention
    # Final Response should now become ready
    # ---------------------------------------------------------

    state["agent_results"][
        RETENTION_AGENT_NAME
    ] = create_retention_result()

    ready_tasks = get_ready_tasks(
        state=state
    )

    ready_agent_names = {
        task.agent_name
        for task in ready_tasks
    }

    assert ready_agent_names == {
        FINAL_RESPONSE_AGENT_NAME
    }

    # ---------------------------------------------------------
    # Complete Final Response
    # Workflow should now be complete
    # ---------------------------------------------------------

    state["agent_results"][
        FINAL_RESPONSE_AGENT_NAME
    ] = create_final_response_result()

    assert is_workflow_complete(
        state=state
    )

    assert get_ready_tasks(
        state=state
    ) == []

    assert get_incomplete_agent_names(
        state=state
    ) == []

    print("PASS")
    print(
        "Prediction + Vector Search "
        "→ Retention → Final Response"
    )
    print()

    # =========================================================
    # TEST 2
    # Simple SQL workflow
    # =========================================================

    print("=" * 80)
    print("TEST 2: Simple SQL workflow")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        create_coordinator_result(
            request_type="sql_analytics",
            execution_plan=[
                AgentTask(
                    task_id="task_1",
                    agent_name=SQL_AGENT_NAME,
                    task_description=(
                        "Count churned customers."
                    ),
                    depends_on=[],
                ),
                AgentTask(
                    task_id="task_2",
                    agent_name=(
                        FINAL_RESPONSE_AGENT_NAME
                    ),
                    task_description=(
                        "Generate the final grounded "
                        "response."
                    ),
                    depends_on=[
                        SQL_AGENT_NAME,
                    ],
                ),
            ],
        )
    )

    ready_tasks = get_ready_tasks(
        state=state
    )

    assert [
        task.agent_name
        for task in ready_tasks
    ] == [
        SQL_AGENT_NAME
    ]

    # We only need a valid successful result for
    # orchestration dependency testing.

    sql_result = SQLAgentResult(
        agent_name=SQL_AGENT_NAME,
        status="success",
        message=(
            "SQL analytics completed successfully."
        ),
        task_description=(
            "Count churned customers."
        ),
        error=None,
        sql_action="count_churned_customers",
        sql_result=[
            {
                "churned_customers": 1869
            }
        ],
    )

    state["agent_results"][
        SQL_AGENT_NAME
    ] = sql_result

    ready_tasks = get_ready_tasks(
        state=state
    )

    assert [
        task.agent_name
        for task in ready_tasks
    ] == [
        FINAL_RESPONSE_AGENT_NAME
    ]

    state["agent_results"][
        FINAL_RESPONSE_AGENT_NAME
    ] = create_final_response_result()

    assert is_workflow_complete(
        state=state
    )

    assert get_incomplete_agent_names(
        state=state
    ) == []

    print("PASS")
    print(
        "SQL Agent → Final Response Agent"
    )
    print()

    # =========================================================
    # TEST 3
    # Blocked workflow
    # =========================================================

    print("=" * 80)
    print("TEST 3: Blocked workflow")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action."
    )

    state["coordinator_result"] = (
        create_coordinator_result(
            request_type="retention",
            execution_plan=[
                AgentTask(
                    task_id="task_1",
                    agent_name=(
                        RETENTION_AGENT_NAME
                    ),
                    task_description=(
                        "Recommend a retention action."
                    ),
                    depends_on=[
                        PREDICTION_AGENT_NAME,
                    ],
                ),
                AgentTask(
                    task_id="task_2",
                    agent_name=(
                        FINAL_RESPONSE_AGENT_NAME
                    ),
                    task_description=(
                        "Generate the final response."
                    ),
                    depends_on=[
                        RETENTION_AGENT_NAME,
                    ],
                ),
            ],
        )
    )

    # Prediction Agent is required but is not part
    # of the execution plan and has no result.

    ready_tasks = get_ready_tasks(
        state=state
    )

    assert ready_tasks == []

    assert not is_workflow_complete(
        state=state
    )

    incomplete_agents = (
        get_incomplete_agent_names(
            state=state
        )
    )

    assert RETENTION_AGENT_NAME in (
        incomplete_agents
    )

    assert FINAL_RESPONSE_AGENT_NAME in (
        incomplete_agents
    )

    print("PASS")
    print(
        "Missing Prediction dependency correctly "
        "blocks the workflow."
    )
    print()

    # =========================================================
    # TEST 4
    # Maximum-iteration protection
    # =========================================================

    print("=" * 80)
    print("TEST 4: Maximum-iteration protection")
    print("=" * 80)

    # ---------------------------------------------------------
    # Mock Coordinator
    # ---------------------------------------------------------

    def mock_coordinator_agent(
        state: MultiAgentState,
        llm_invoke: LLMInvokeFunction,
    ) -> MultiAgentState:
        """
        Create a simple SQL execution plan.
        """

        state["coordinator_result"] = (
            create_coordinator_result(
                request_type="sql_analytics",
                execution_plan=[
                    AgentTask(
                        task_id="task_1",
                        agent_name=SQL_AGENT_NAME,
                        task_description=(
                            "Count churned customers."
                        ),
                        depends_on=[],
                    ),
                    AgentTask(
                        task_id="task_2",
                        agent_name=(
                            FINAL_RESPONSE_AGENT_NAME
                        ),
                        task_description=(
                            "Generate the final response."
                        ),
                        depends_on=[
                            SQL_AGENT_NAME,
                        ],
                    ),
                ],
            )
        )

        return state

    # ---------------------------------------------------------
    # Mock runner that never stores a successful result
    # ---------------------------------------------------------

    def mock_non_completing_sql_agent(
        state: MultiAgentState,
        sql_analytics_tool: Any,
    ) -> MultiAgentState:
        """
        Simulate an agent that returns Shared State
        without completing its task.
        """

        return state

    def mock_unused_tool(
        *args: Any,
        **kwargs: Any,
    ) -> Dict[str, Any]:

        return {
            "status": "success"
        }

    def mock_llm(
        prompt: str,
    ) -> str:

        return "Mock response."

    # ---------------------------------------------------------
    # We test the iteration guard with a small local
    # orchestration loop so no real service is called.
    # ---------------------------------------------------------

    state = create_initial_state(
        "How many customers churned?"
    )

    state = mock_coordinator_agent(
        state=state,
        llm_invoke=mock_llm,
    )

    mock_agent_runners = {
        SQL_AGENT_NAME: (
            mock_non_completing_sql_agent,
            {
                "sql_analytics_tool": (
                    mock_unused_tool
                ),
            },
        ),
    }

    max_iterations = 2
    iteration = 0
    maximum_iterations_reached = False

    while not is_workflow_complete(
        state=state
    ):

        iteration += 1

        if iteration > max_iterations:
            maximum_iterations_reached = True
            break

        ready_tasks = get_ready_tasks(
            state=state
        )

        assert ready_tasks

        for task in ready_tasks:

            if (
                task.agent_name
                not in mock_agent_runners
            ):
                continue

            state = run_agent(
                state=state,
                task=task,
                agent_runners=(
                    mock_agent_runners
                ),
            )

    assert maximum_iterations_reached

    assert iteration == (
        max_iterations + 1
    )

    assert not is_workflow_complete(
        state=state
    )

    print("PASS")
    print(
        "Maximum-iteration guard prevented "
        "an infinite orchestration loop."
    )
    print()

    # =========================================================
    # Final result
    # =========================================================

    print("=" * 80)
    print(
        "ALL MULTI-AGENT ORCHESTRATOR "
        "UNIT TESTS PASSED"
    )
    print("=" * 80)

##### 23. Expected Results

- The Coordinator creates a valid execution plan.

- Only unfinished tasks are considered.

- Tasks execute only after their dependencies succeed.

- Each agent stores its validated result in shared state.
 
- Each completed task is skipped in future iterations.

- The workflow stops when all planned tasks complete.

- The Final Response Agent creates a grounded response.

- Blocked or invalid workflows produce clear errors.

##### 24. Key Learnings

###### Shared state coordinates the workflow

- All agents read from and write to the same MultiAgentState.

- This allows downstream agents to use earlier results without calling earlier agents directly.

###### The execution plan controls agent order

The orchestrator does not hard-code one fixed workflow.

It follows: ``` text state["execution_plan"] ``` created by the Coordinator Agent.


###### Dependencies determine readiness

A task can execute only when all agents in: ``` text task.depends_on ``` have successful results.


###### The orchestrator is not another AI agent

- The orchestrator contains deterministic Python logic.

- It:

    - checks completion
    - checks dependencies
    - finds ready tasks
    - runs agents
    - detects blocked workflows

- It does not make business decisions.

- The registry separates configuration from execution

- AGENT_RUNNERS stores:

    - agent name
    - agent function
    - required arguments

- The generic run_agent() function can therefore execute agents with different function signatures.

###### Individual agents remain responsible for their own work

- Each agent continues to handle:

    - tool execution
    - output validation
    - result storage
    - execution-history updates
    - agent-level error recording

- The orchestrator does not duplicate that logic.

##### 25. Notebook Conclusion

- This notebook completes the end-to-end multi-agent customer-support workflow.

- The final architecture now contains:

    - Coordinator Agent
    - SQL Agent
    - Prediction Agent
    - Vector Search Agent
    - Retention Agent
    - Final Response Agent
    - Dependency-based Orchestrator
    - Shared Multi-Agent State
    - Validated Agent Schemas
    - Execution History
    - Error Tracking

- The Coordinator decides which agents are needed and creates the execution plan.

- The orchestrator reads that plan, checks task dependencies, executes ready agents, prevents duplicate execution, detects blocked workflows, and continues until all planned work is complete.

- Each specialized agent remains independent and responsible for its own business logic.

- The final result is a modular and extensible multi-agent system that can support SQL analytics, churn prediction, semantic customer-note retrieval, retention recommendations, and grounded customer responses.